# 01 — Vector Addition

**Ladder:** Naive (1 element / pid) → blocked + masked → Triton `tl` kernel.

**Study tip.** Master `offsets` + `mask` on a tiny $N$ not divisible by $B$ before any benchmark. This folder is the SPMD foundation for everything else.

Official: [Vector Addition](https://triton-lang.org/main/getting-started/tutorials/01-vector-add.html)

Figures: [`docs/images/`](docs/images/) · visual guide: [`docs/visual-guide.md`](docs/visual-guide.md)

---

## 0. Definition

Let $x,y \in \mathbb{R}^{N}$. Define the elementwise sum $z \in \mathbb{R}^{N}$ by

$$
z_i = x_i + y_i, \qquad i = 0,1,\ldots,N-1.
$$

**Claim (correctness).** The map $(x,y) \mapsto z$ is componentwise addition on $\mathbb{R}^{N}$. For any index set $I \subseteq \{0,\ldots,N-1\}$,

$$
\forall i \in I: \quad z_i = x_i + y_i
$$

is necessary and sufficient for $z$ to equal $x+y$ on $I$.

**Arithmetic intensity.** Each output costs $1$ FLOP and $3$ memory accesses (load $x_i$, load $y_i$, store $z_i$). With element size $b$ bytes and runtime $t$ seconds,

$$
\mathrm{GB/s} = \frac{3 N b}{t \cdot 10^{9}}.
$$

Hence the kernel is **memory-bound**: performance tracks DRAM bandwidth, not peak FLOPs.

**Running example (masking).** $N=5$, $B=4$ → $P=\lceil 5/4\rceil=2$ programs; pid $1$ only writes index $4$.

**Teaching / bench sizes.** Pseudo tests use $N=1000$ with $B\in\{128,256\}$; GPU correctness uses $N=98432$; official bench sweeps $N=2^{12}\ldots 2^{27}$.

### Hardware mental model (before kernels)

![Grid → Thread Block → Warp → Threads](docs/images/grid-block-warp-thread.jpeg)

![CUDA hierarchy: grid, blocks, warps of 32, SIMT](docs/images/cuda-execution-hierarchy.png)

Triton **programs** correspond to CUDA blocks. Vector-add launches a 1-D grid; each program owns either one index (naive) or one contiguous tile of length $B$ (blocked).

![GPU Computing Fundamentals cheat sheet](docs/images/gpu-fundamentals-cheatsheet.png)

*Primer: CPU vs GPU, warps, coalescing, fusion — vector-add is the textbook **memory-bound** / embarrassingly parallel case.*


### Visual: theory board (method / logic)

![Whiteboard: naive 1-element programs vs blocked + mask tiling](docs/images/benchmark-vector-add.png)

*Theory board — naive grid size $N$ vs blocked tiles with `mask = offsets < N` (same algebra as the Triton kernel).*

### Theory board

![Whiteboard: naive 1-element programs vs blocked + mask tiling](docs/images/whiteboard-theory.jpg)

*Theory board — naive grid size $N$ vs blocked tiles with `mask = offsets < N` (same algebra as the Triton kernel).*

### Pitfalls

- Forgetting the mask on the last tile → out-of-bounds loads.
- Using BLOCK_SIZE that is not a power of two can hurt Triton codegen; prefer 128/256/1024.
- Measuring tiny N → launch overhead dominates; bench large N for GB/s.

### Self-check

1. Write offsets and mask for N=5, B=4 by hand.
2. Why is vector-add memory-bound? Count bytes moved vs FLOPs.
3. Change B and re-run: does correctness still hold on a remainder size?


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pytest
import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



## 1. Naive map — one program per index

**Construction.** Identify each program id with an index:

$$
\mathrm{pid} = i \in \{0,\ldots,N-1\}, \qquad z_i \leftarrow x_i + y_i.
$$

**Proof of coverage.** The set of pids is $\{0,\ldots,N-1\} = \mathrm{dom}(z)$, so every coordinate is written exactly once. Uniqueness of the write follows because $\mathrm{pid} \mapsto i$ is bijective.

**Concrete check.** For $N=8$ there are $8$ programs; pid $5$ writes $z_5 = x_5 + y_5$.

**Cost of this schedule.** Grid size $= N$. Launching $N$ independent programs for one FLOP each is correct but inefficient: occupancy and launch overhead dominate — motivating the blocked schedule next.


In [ ]:
def naive_add_pseudo(x, y):
    """
    Elementwise add with one "pid" per output element.
    (See markdown cells above for the math.)
    """
    assert x.shape == y.shape
    n = x.numel()
    z = torch.empty_like(x)

    # Flatten so indexing mirrors the 1-D Triton grid
    x_flat = x.reshape(-1)
    y_flat = y.reshape(-1)
    z_flat = z.reshape(-1)

    for pid in range(n):
        z_flat[pid] = x_flat[pid] + y_flat[pid]

    return z


## 2. Blocking + masking (tiling the 1-D domain)

Fix a block size $B \in \mathbb{N}_{\ge 1}$. Partition $\{0,\ldots,N-1\}$ into contiguous tiles

$$
T_p = \{ pB + j \mid 0 \le j < B \} \cap \{0,\ldots,N-1\},
\qquad p = 0,\ldots,P-1,
$$

where the number of programs is

$$
P = \left\lceil \frac{N}{B} \right\rceil.
$$

**Offsets (before masking).** Program $p$ owns the arithmetic progression

$$
\mathrm{offsets}(p) = pB + \{0,1,\ldots,B-1\}.
$$

**Mask (predicate for validity).**

$$
m_j = \mathbf{1}\big[\mathrm{offsets}(p)_j < N\big].
$$

**Teaching walk** ($N=5$, $B=4$ — same remainder case as in `blocked_add_pseudo` comments):

| pid | offsets (pre-mask) | mask | writes |
|-----|--------------------|------|--------|
| 0 | $\{0,1,2,3\}$ | all true | $z[0:4]$ |
| 1 | $\{4,5,6,7\}$ | only index 4 | $z[4]$ |

**Lemma (no OOB).** If we only load/store where $m_j = 1$, then every accessed index lies in $\{0,\ldots,N-1\}$.

*Proof.* Immediate from the definition of $m_j$.

**Lemma (partition).** $\bigcup_p T_p = \{0,\ldots,N-1\}$ and $T_p \cap T_{p'} = \emptyset$ for $p \neq p'$.

*Proof.* Each $i$ belongs to tile $p = \lfloor i/B \rfloor$ with local offset $i \bmod B$; uniqueness of quotient/remainder gives disjointness.

**Corollary.** Computing $z_i = x_i + y_i$ for all $i \in T_p$, over all $p$, yields the unique correct $z$.

Typical host defaults: $B=1024$, `grid = cdiv(N, B)`.


In [ ]:
def blocked_add_pseudo(x, y, BLOCK_SIZE: int = 1024):
    """
    Each pid owns a contiguous block of BLOCK_SIZE elements.
    (See markdown cells above for the math.)
    """
    assert x.shape == y.shape
    n = x.numel()
    z = torch.empty_like(x)

    x_flat = x.reshape(-1)
    y_flat = y.reshape(-1)
    z_flat = z.reshape(-1)

    num_pids = (n + BLOCK_SIZE - 1) // BLOCK_SIZE  # ceil division
    for pid in range(num_pids):
        block_start = pid * BLOCK_SIZE
        offsets = [block_start + i for i in range(BLOCK_SIZE)]

        for off in offsets:
            if off < n:  # mask for the last (partial) block
                z_flat[off] = x_flat[off] + y_flat[off]

    return z


## 3. Triton realization (`tl`)

The blocked construction becomes SPMD code:

$$
\begin{aligned}
\mathrm{pid} &= \mathtt{tl.program\_id}(0),\\
\mathrm{offsets} &= \mathrm{pid}\cdot B + \mathtt{tl.arange}(0,B),\\
m &= \mathrm{offsets} < N,\\
z[\mathrm{offsets}] &\leftarrow x[\mathrm{offsets}] + y[\mathrm{offsets}]
\quad\text{(masked by $m$)}.
\end{aligned}
$$

**Equivalence.** For every legal index $i$, there exist unique $p,j$ with $i=pB+j$ and $m_j=1$, so the store writes exactly $x_i+y_i$. Thus the Triton kernel realizes the same function as the blocked pseudocode, which realizes the same function as the naive map.

```
|-- block 0 --|-- block 1 --|-- ... --| rem |
     pid=0         pid=1
```

| Kernel / stage | Program owns | Indexing |
|---|---|---|
| `naive_add_pseudo` | one $z_i$ | $\mathrm{pid}=i$ |
| `blocked_add_pseudo` | tile of $B$ | $pB+\{0..B-1\}$ + mask |
| `add_kernel` | same tile | `tl.program_id` / masked load-store |

**Bandwidth check.** After correctness (`allclose` vs `torch.add`), bench GB/s with $3Nb/(t\cdot 10^9)$ — expect Triton ≈ PyTorch near peak DRAM.


In [ ]:
@triton.jit
def add_kernel(
    x_ptr,
    y_ptr,
    output_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    # Same blocked idea as blocked_add_pseudo:
    #   pid -> block_start -> offsets -> mask -> load / add / store
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    tl.store(output_ptr + offsets, x + y, mask=mask)


def add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    output = torch.empty_like(x)
    n_elements = output.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    return output


In [ ]:
def test_pseudo_matches_torch() -> None:
    """CPU-runnable: naive and blocked Python stages match torch.add."""
    torch.manual_seed(0)
    x = torch.rand(1000)
    y = torch.rand(1000)
    ref = x + y
    assert torch.allclose(naive_add_pseudo(x, y), ref)
    assert torch.allclose(blocked_add_pseudo(x, y, BLOCK_SIZE=128), ref)
    # Remainder case: n not divisible by BLOCK_SIZE
    assert torch.allclose(blocked_add_pseudo(x, y, BLOCK_SIZE=256), ref)


def test_correctness() -> None:
    require_gpu()
    torch.manual_seed(0)
    size = 98432
    x = torch.rand(size, device=DEVICE)
    y = torch.rand(size, device=DEVICE)
    out_torch = x + y
    out_triton = add(x, y)
    assert torch.allclose(out_torch, out_triton), torch.max(torch.abs(out_torch - out_triton)).item()
    print("correctness OK", torch.max(torch.abs(out_torch - out_triton)).item())


In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["size"],
        x_vals=[2**i for i in range(12, 28)],
        x_log=True,
        line_arg="provider",
        line_vals=["triton", "torch"],
        line_names=["Triton", "Torch"],
        styles=[("blue", "-"), ("green", "-")],
        ylabel="GB/s",
        plot_name="vector-add-performance",
        args={},
    )
)
def benchmark(size, provider):
    x = torch.rand(size, device=DEVICE, dtype=torch.float32)
    y = torch.rand(size, device=DEVICE, dtype=torch.float32)
    quantiles = [0.5, 0.2, 0.8]
    if provider == "torch":
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: x + y, quantiles=quantiles)
    if provider == "triton":
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: add(x, y), quantiles=quantiles)
    gbps = lambda ms: 3 * x.numel() * x.element_size() * 1e-9 / (ms * 1e-3)
    return gbps(ms), gbps(max_ms), gbps(min_ms)
